<a href="https://colab.research.google.com/github/PARIMIANUDHEER04/PyTorch/blob/main/Vision_Tranformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
class PatchEmbed(nn.Module):
  def __init__(self,inp=224,patch_size=16,in_channel=3,embed_size=768) -> None:
    super().__init__()
    self.num_patches=inp//patch_size * inp//patch_size
    self.conv=nn.Conv2d(in_channels=in_channel,out_channels=embed_size,kernel_size=patch_size,stride=patch_size)
  def forward(self,x):
    out=self.conv(x)
    return out.flatten(2).transpose(1,2)



In [3]:
class MLP(nn.Module):
  def __init__(self,inp1,inp2,dropout=0.0):
    super().__init__()
    self.fc1=nn.Linear(inp1,inp2,bias=True)
    self.bn1=nn.GELU()
    self.fc2=nn.Linear(inp2,inp1,bias=True)
    self.dropout=nn.Dropout(dropout)
  def forward(self,x):
    out=self.fc1(x)
    out=self.bn1(out)
    out=self.fc2(out)
    return self.dropout(out)

In [4]:
class TransformerBlock(nn.Module):
  def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True,attn_dropout=0.0, proj_dropout=0.0):
    super().__init__()
    self.norm1=nn.LayerNorm(dim,eps=1e-6)
    self.attn = nn.MultiheadAttention(embed_dim=dim,num_heads=num_heads,dropout=attn_dropout,bias=qkv_bias,batch_first=True)
    self.dropout1 = nn.Dropout(proj_dropout)
    self.norm2 = nn.LayerNorm(dim, eps=1e-6)
    hidden_dim = int(dim * mlp_ratio)
    self.mlp = MLP(dim, hidden_dim, dropout=proj_dropout)
  def forward(self, x):
    y = self.norm1(x)
    attn_out, _ = self.attn(y, y, y, need_weights=False)
    x= x+self.dropout1(attn_out)
    x= x+self.mlp(self.norm2(x))
    return x


In [5]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        num_classes=1000,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        qkv_bias=True,
        dropout=0.0,
        emb_dropout=0.0
    ):
        super().__init__()

        self.patch_embed = PatchEmbed(
            img_size, patch_size, in_chans, embed_dim
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1,1+num_patches, embed_dim))
        self.pos_drop = nn.Dropout(emb_dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                attn_dropout=dropout,
                proj_dropout=dropout
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.head = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x, return_features=False):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        cls_out = x[:, 0]
        if return_features:
            return cls_out, x
        return self.head(cls_out)

In [6]:
# CIFAR-10 has 32x32 images, so we'll resize to 224x224 for ViT
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


100%|██████████| 170M/170M [00:11<00:00, 15.0MB/s]


In [7]:
# Your VisionTransformer class is already defined
num_classes = 10
model = VisionTransformer(
    img_size=224,
    patch_size=16,
    in_chans=3,
    num_classes=num_classes,
    embed_dim=768,
    depth=12,
    num_heads=12,
    mlp_ratio=4.0,
    dropout=0.1,
    emb_dropout=0.0
).to(device)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)


In [9]:
epochs = 5  # for demo, increase for real training

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/5], Loss: 2.3225
Epoch [2/5], Loss: 2.2242
Epoch [3/5], Loss: 2.1417
Epoch [4/5], Loss: 2.1920
Epoch [5/5], Loss: 2.1562


In [10]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


Test Accuracy: 19.11%
